# 🤖 Fußball Match Outcome Predictor – Modelltraining

**Projekt:** KI & Intelligence Engineering  
**Institution:** DHBW Mannheim  

## 📌 Beschreibung

Dieses Skript bildet die Trainingspipeline für den Fußball-Match-Outcome-Predictor.  
Es verwendet die zuvor aufbereiteten Daten aus der Datenpipeline, trainiert verschiedene Machine-Learning-Modelle und bewertet deren Performance.

Die Pipeline umfasst folgende Schritte:

1. **Datenimport**
   - Lädt den vorbereiteten Datensatz aus `datenpipeline.py`.
   - Bereitet die Daten für das Modelltraining vor.

2. **Modelltraining**
   - Trainiert mehrere Klassifikationsmodelle:
     - **XGBoost**
     - **Random Forest**
     - **Logistic Regression**

3. **Hyperparameter-Optimierung**
   - Führt automatisiertes Tuning mit `GridSearchCV` durch.
   - Sucht die optimalen Modellparameter für eine bessere Vorhersageleistung.

4. **Modellvergleich**
   - Bewertet die Modelle anhand verschiedener Metriken:
     - Accuracy
     - F1-Score (Macro)
     - Log Loss

5. **Modellspeicherung**
   - Speichert die besten trainierten Modelle für die spätere Nutzung.

---

## 🚀 Verwendung

Das Skript kann direkt über die Kommandozeile ausgeführt werden:

```bash
python modelltraining.ipynb


**imports**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss, confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
import xgboost as xgb
import warnings
import joblib
import os

warnings.filterwarnings('ignore')

# Config

In [2]:
DATA_DIR = "data/processed"
MODEL_DIR = "models"

print("=" * 60)
print("🤖  Fußball Match Outcome Predictor – Modelltraining")
print("=" * 60)

🤖  Fußball Match Outcome Predictor – Modelltraining


# Daten Laden

In [3]:
print("" + "─" * 60)
print("📂 Schritt 1: Daten laden")
print("─" * 60)

train_df = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=['date'])
val_df = pd.read_csv(f"{DATA_DIR}/val.csv", parse_dates=['date'])
test_df = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=['date'])
features_df = pd.read_csv(f"{DATA_DIR}/features.csv")

feature_cols = features_df['feature'].tolist()
target_col = 'target'

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Features: {len(feature_cols)}")

────────────────────────────────────────────────────────────
📂 Schritt 1: Daten laden
────────────────────────────────────────────────────────────
Train: 316,517 | Val: 3,540 | Test: 21,092
Features: 14


# Feature Matrix 

In [4]:
print("" + "─" * 60)
print("🔧 Schritt 2: Feature-Matrix vorbereiten")
print("─" * 60)

# Kombiniere Train + Val für Cross-Validation
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

X_train_val = train_val_df[feature_cols]
y_train_val = train_val_df[target_col]

print(f"Train:      X={X_train.shape}, y={y_train.shape}")
print(f"Val:        X={X_val.shape}, y={y_val.shape}")
print(f"Test:       X={X_test.shape}, y={y_test.shape}")
print(f"Train+Val:  X={X_train_val.shape}, y={y_train_val.shape}")

────────────────────────────────────────────────────────────
🔧 Schritt 2: Feature-Matrix vorbereiten
────────────────────────────────────────────────────────────
Train:      X=(316517, 14), y=(316517,)
Val:        X=(3540, 14), y=(3540,)
Test:       X=(21092, 14), y=(21092,)
Train+Val:  X=(320057, 14), y=(320057,)


# Preprocessing

In [5]:
print("" + "─" * 60)
print("⚙️  Schritt 3: Preprocessing")
print("─" * 60)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), feature_cols)
])

print("✅ StandardScaler für numerische Features")

────────────────────────────────────────────────────────────
⚙️  Schritt 3: Preprocessing
────────────────────────────────────────────────────────────
✅ StandardScaler für numerische Features


# Modell 1: Dummy Classifier (baseline)

In [7]:
print("" + "─" * 60)
print("📊 Modell 1: Dummy Classifier (Baseline)")
print("─" * 60)

dummy = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent', random_state=42))
])

dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)

acc_dummy = accuracy_score(y_test, y_pred_dummy)
f1_dummy = f1_score(y_test, y_pred_dummy, average='macro')
ll_dummy = log_loss(y_test, y_proba_dummy)

print(f"Accuracy:  {acc_dummy:.4f}")
print(f"  F1-Macro:  {f1_dummy:.4f}")
print(f"  Log Loss:  {ll_dummy:.4f}")

────────────────────────────────────────────────────────────
📊 Modell 1: Dummy Classifier (Baseline)
────────────────────────────────────────────────────────────
Accuracy:  0.4950
  F1-Macro:  0.2207
  Log Loss:  18.2030


# Modell 2: Logistic Regression

In [8]:
print("" + "─" * 60)
print("📊 Modell 2: Logistic Regression")
print("─" * 60)

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'))
])

lr_params = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0],
    'classifier__solver': ['lbfgs', 'saga']
}

print("🔍 GridSearch...")
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
lr_grid.fit(X_train_val, y_train_val)

print(f"✅ Beste Parameter: {lr_grid.best_params_}")
print(f"Beste F1-Macro (CV): {lr_grid.best_score_:.4f}")

y_pred_lr = lr_grid.predict(X_test)
y_proba_lr = lr_grid.predict_proba(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='macro')
ll_lr = log_loss(y_test, y_proba_lr)

print(f"📊 Logistic Regression (Test):")
print(f"  Accuracy:  {acc_lr:.4f}")
print(f"  F1-Macro:  {f1_lr:.4f}")
print(f"  Log Loss:  {ll_lr:.4f}")

────────────────────────────────────────────────────────────
📊 Modell 2: Logistic Regression
────────────────────────────────────────────────────────────
🔍 GridSearch...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
✅ Beste Parameter: {'classifier__C': 0.1, 'classifier__solver': 'saga'}
Beste F1-Macro (CV): 0.5271
📊 Logistic Regression (Test):
  Accuracy:  0.4802
  F1-Macro:  0.3765
  Log Loss:  1.6099


# Modell 3: Random Forest

In [9]:
print("" + "─" * 60)
print("🌲 Modell 3: Random Forest")
print("─" * 60)

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

print("🔍 GridSearch...")
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
rf_grid.fit(X_train_val, y_train_val)

print(f"✅ Beste Parameter: {rf_grid.best_params_}")
print(f"Beste F1-Macro (CV): {rf_grid.best_score_:.4f}")

y_pred_rf = rf_grid.predict(X_test)
y_proba_rf = rf_grid.predict_proba(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='macro')
ll_rf = log_loss(y_test, y_proba_rf)

print(f"📊 Random Forest (Test):")
print(f"  Accuracy:  {acc_rf:.4f}")
print(f"  F1-Macro:  {f1_rf:.4f}")
print(f"  Log Loss:  {ll_rf:.4f}")

────────────────────────────────────────────────────────────
🌲 Modell 3: Random Forest
────────────────────────────────────────────────────────────
🔍 GridSearch...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
✅ Beste Parameter: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Beste F1-Macro (CV): 0.5060
📊 Random Forest (Test):
  Accuracy:  0.4677
  F1-Macro:  0.3600
  Log Loss:  1.0734
